# 04 · OOP Core — **Depth**

> **Pull this notebook when:** you start writing classes for real — transformer modules, the mood
> model, memory stores. The mechanics of attribute lookup and method binding decide whether your
> classes behave or surprise you.

Depth here is the **attribute lookup model** and what methods/properties actually are. Predict first.

---

## 4.1 — Predict: the shared class-attribute bug

A class attribute is shared by all instances; an instance attribute belongs to one object. Watch what
that means when the class attribute is **mutable**.

```python
class Dog:
    tricks = []                  # class attribute
    def __init__(self, name):
        self.name = name
    def add_trick(self, t):
        self.tricks.append(t)    # mutates the shared list

a = Dog("Rex"); b = Dog("Fido")
a.add_trick("sit")
b.add_trick("roll")
```
What is `a.tricks`?

In [ ]:
class Dog:
    tricks = []
    def __init__(self, name):
        self.name = name
    def add_trick(self, t):
        self.tricks.append(t)

a = Dog("Rex"); b = Dog("Fido")
a.add_trick("sit")
b.add_trick("roll")
print(a.tricks)   # ?
print(b.tricks)   # ?

<details>
<summary>▶ Reveal</summary>

Both print `['sit', 'roll']`.

`tricks = []` lives on the *class*, so there's **one list shared by every instance**. `self.tricks.append`
doesn't create a per-instance list — it mutates the single shared one (attribute *lookup* finds it on
the class; `.append` mutates it in place). Rex and Fido see each other's tricks.

The fix: make it an *instance* attribute by assigning in `__init__` — `self.tricks = []` — so each Dog
gets its own list. Use class attributes only for things genuinely shared (constants, counters); never
for mutable per-instance state. This is the OOP cousin of the mutable-default-argument trap.

</details>

## 4.2 — Predict: instance vs class attribute shadowing

Attribute lookup checks the **instance** `__dict__` first, then the **class**. Assigning to
`self.x` creates an instance attribute that *shadows* the class one — for that instance only.

```python
class C:
    x = "class"
a = C(); b = C()
a.x = "instance"      # creates instance attr on a only
print(a.x)            # ?
print(b.x)            # ?
print(C.x)            # ?
```

In [ ]:
class C:
    x = "class"
a = C(); b = C()
a.x = "instance"
print(a.x)
print(b.x)
print(C.x)

<details>
<summary>▶ Reveal</summary>

`instance`, `class`, `class`.

`a.x = "instance"` writes into `a`'s own `__dict__`. Looking up `a.x` finds it there first, so it
shadows the class attribute — but *only for `a`*. `b` has nothing in its own `__dict__`, so `b.x`
falls through to the class and finds `"class"`. `C.x` is untouched. Lookup order is **instance →
class → base classes**; the first hit wins. Contrast with 4.1: there you *mutated* the shared object
(visible to all); here you *rebound* the name on one instance (visible to one). Mutate vs rebind, again.

</details>

## 4.3 — Predict: what is a method, really?

```python
class Greeter:
    def hi(self):
        return "hi"

g = Greeter()
print(type(g.hi))           # ? (accessed via instance)
print(type(Greeter.hi))     # ? (accessed via class)
print(Greeter.hi(g))        # ? (call the class's function, pass instance manually)
```

In [ ]:
class Greeter:
    def hi(self):
        return "hi"

g = Greeter()
print(type(g.hi).__name__)
print(type(Greeter.hi).__name__)
print(Greeter.hi(g))

<details>
<summary>▶ Reveal</summary>

`method`, `function`, `hi`.

`Greeter.hi` is just a **plain function** stored on the class. `g.hi` is a **bound method** — accessing
the function *through an instance* binds `self` to that instance automatically. That's the entire
mystery of `self`: `g.hi()` is really `Greeter.hi(g)` with `g` slotted in as `self`. The last line
proves it — calling the raw function and passing the instance by hand works identically. `self` isn't
magic; it's the instance, passed as the first argument, made automatic by the binding that happens on
attribute access.

</details>

## 4.4 — Predict: an attribute that runs code

```python
class Circle:
    def __init__(self, r):
        self.r = r
    @property
    def area(self):
        print("  (computing area)")
        return 3.14159 * self.r ** 2

c = Circle(2)
print("before access")
x = c.area            # note: no parentheses
print("after access")
```
When does `(computing area)` print — and does `c.area` need `()`?

In [ ]:
class Circle:
    def __init__(self, r):
        self.r = r
    @property
    def area(self):
        print("  (computing area)")
        return 3.14159 * self.r ** 2

c = Circle(2)
print("before access")
x = c.area
print("after access")
print(x)

<details>
<summary>▶ Reveal</summary>

```
before access
  (computing area)
after access
12.56636
```

`c.area` — **no parentheses** — runs the method body. `@property` turns `area` into something accessed
like a plain attribute, but reading it *calls the getter function*. (Under the hood `property` is a
*descriptor*: an object whose `__get__` runs when you access the attribute. You don't need to implement
descriptors, but knowing a property is "a method that looks like an attribute" explains why `c.area`
executes code and why you must not write `c.area()`.) Use it to expose computed or validated values
with attribute syntax.

</details>

## 4.5 — Three ways to expose a derived value (and when each)

`full_name` from `first` and `last` can be: a **method** (`p.full_name()`), a **property**
(`p.full_name`), or a **stored attribute** set in `__init__`. Implement all three on three small
classes and feel the trade-offs (answer explains when to use which).

(build all three below)

In [ ]:
class PersonMethod:
    def __init__(self, first, last):
        self.first, self.last = first, last
    def full_name(self):
        # YOUR CODE HERE
        pass

class PersonProperty:
    def __init__(self, first, last):
        self.first, self.last = first, last
    @property
    def full_name(self):
        # YOUR CODE HERE
        pass

class PersonStored:
    def __init__(self, first, last):
        self.first, self.last = first, last
        # YOUR CODE HERE — compute and store self.full_name now
        pass

# Tests
assert PersonMethod("Ada", "Lovelace").full_name() == "Ada Lovelace"   # called
assert PersonProperty("Ada", "Lovelace").full_name == "Ada Lovelace"   # accessed
assert PersonStored("Ada", "Lovelace").full_name == "Ada Lovelace"     # stored
print("4.5 passed")

<details>
<summary>▶ When each</summary>

- **Property** — the default for a *derived* value that should read like data and always reflect
  current state. `PersonProperty.full_name` recomputes from `first`/`last` every access, so changing
  `first` updates it. Clean call site, always correct.
- **Method** — when the operation is clearly an *action* or takes arguments, or is expensive enough
  that you want the `()` to signal "work happens here."
- **Stored** — only when the value is fixed at construction and recomputing is wasteful. Danger: it
  goes *stale* if `first` changes later, because it was computed once. Use only for genuinely immutable
  derived data.

The recurring judgment: does it need to stay in sync with changing state? Property. Is it a one-time
snapshot? Stored. Is it an action? Method.

</details>

## ★ Milestone 04 — Two properties kept in sync

Synthesize properties + validation + an alternative constructor. Build `Temperature` storing celsius
internally, exposing **both** `celsius` and `fahrenheit` as properties with getters *and setters*, so
setting either keeps the other consistent. Add a `classmethod` `from_fahrenheit(f)` as an alternative
constructor, and reject below absolute zero (−273.15 °C) with `ValueError`.

This is the mood-model shape from Week 7 in miniature: an object that exposes derived views and
protects its own invariants.

(build it below)

Property setters use the `@<name>.setter` decorator:
```python
@celsius.setter
def celsius(self, value): ...
```

In [ ]:
class Temperature:
    def __init__(self, celsius=0.0):
        self.celsius = celsius          # goes through the setter (validation)

    @property
    def celsius(self):
        # YOUR CODE HERE
        pass

    @celsius.setter
    def celsius(self, value):
        # YOUR CODE HERE — reject < -273.15 with ValueError, else store
        pass

    @property
    def fahrenheit(self):
        # YOUR CODE HERE — derive from celsius
        pass

    @fahrenheit.setter
    def fahrenheit(self, value):
        # YOUR CODE HERE — convert to celsius and store (reuse the celsius setter)
        pass

    @classmethod
    def from_fahrenheit(cls, f):
        # YOUR CODE HERE
        pass

# Tests
t = Temperature(100)
assert t.celsius == 100
assert abs(t.fahrenheit - 212) < 1e-9          # derived view
t.fahrenheit = 32                               # set one...
assert abs(t.celsius - 0) < 1e-9                # ...the other stays in sync
t2 = Temperature.from_fahrenheit(212)           # alt constructor
assert abs(t2.celsius - 100) < 1e-9
assert isinstance(t2, Temperature)
raised = False
try:
    Temperature(-300)                           # invariant enforced
except ValueError:
    raised = True
assert raised
print("Milestone 04 passed")

<details>
<summary>▶ How it stays consistent</summary>

There's **one** piece of stored state — `self._celsius`. `fahrenheit` is a pure derived view (getter
computes from celsius; setter converts back and routes through `self.celsius = ...`, which re-runs the
validation). So there's no way for the two to disagree, because fahrenheit isn't stored at all — it's
always computed from the single source of truth. Validation lives in the celsius setter, so *every*
path that changes temperature (init, celsius=, fahrenheit=, from_fahrenheit) is guarded by it. That
"one source of truth, derived views, validation at the choke point" design is exactly what keeps Mara's
mood dimensions valid no matter what updates them.

</details>